[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/28.%20The%20Integrated%20Crane%20Assignment%20%26%20Scheduling%20Problem%20%28QCAP-QCSP%29/P28-Tier-2_executed.ipynb)

---


# 28. The Integrated Crane Assignment and Scheduling Problem
## Tier 2: A Constructive EAC-SPT Heuristic

### Goal and inputs

This notebook implements the Earliest Available Crane–Shortest Processing Time (EAC-SPT) heuristic for assigning bay tasks to quay cranes and constructing a schedule. Its inputs are vessels and their bay tasks, cranes with initial berth positions, and a minimum clearance distance. Its outputs are task assignments, a schedule for each crane, the makespan, performance metrics, and visualizations.

A *constructive heuristic* builds a solution one decision at a time. EAC-SPT first prioritizes tasks by processing time, then assigns each task to the crane that can start it earliest without violating the modeled spatial constraints. This is a greedy rule: each assignment is chosen using the information available at that step, without searching all possible schedules. It is therefore a practical construction method, not a proof of global optimality.

### Model assumptions and scope

- Processing times, start times, and crane availability times are nonnegative integers. Each task occupies its assigned crane on the half-open interval $[s, s+p)$, where $s$ is its start time and $p$ its processing time.
- A crane can process only one task at a time. Tasks are available from time zero; vessel arrival times and due dates are stored in the data but are not used by this heuristic.
- A crane's left-to-right order is determined by its initial berth position. That order cannot be crossed while two cranes are serving overlapping tasks.
- During overlapping tasks, assigned bay positions must be separated by at least the specified clearance distance. Travel time, crane movement between tasks, setup times, and uncertainty are not modeled.
- The example uses zero-based IDs. The implementation uses crane IDs to identify cranes and vessels, and the visualization assumes vessel IDs run from zero to the number of vessels minus one.

These assumptions define the scheduling model solved here; the resulting schedule should be interpreted within this scope.

### EAC-SPT construction rule

Let $J$ be the set of bay tasks, $C$ the set of cranes, and $p_j$ the processing time of task $j$. The SPT rule orders tasks by nondecreasing processing time. For each task in that order, the heuristic finds the earliest feasible start time $t_{cj}$ for every crane $c$ and selects the crane with the smallest such time:

$$
c^*(j) \in \operatorname*{arg\,min}_{c \in C} t_{cj}.
$$

The feasible start time $t_{cj}$ must be no earlier than crane $c$'s availability and must satisfy the spatial constraints throughout the candidate task's processing interval. If several cranes have the same earliest feasible time, the priority queue's crane-ID order breaks the tie. If task processing times tie, their input order breaks the tie.

For tasks $i$ and $j$ on different cranes, their occupied intervals overlap exactly when $s_i < s_j+p_j$ and $s_j < s_i+p_i$. When they overlap, the model requires both sufficient berth clearance, $|x_i-x_j| \ge d$, and preservation of crane order. Here, $x_i$ and $x_j$ are bay positions and $d$ is the clearance distance. The half-open interval convention allows one task to start at the exact time another ends.

The schedule's makespan is $C_{\max}=\max_{j\in J}(s_j+p_j)$, the completion time of the last task. The heuristic does not minimize this quantity by exhaustive search; it reports the makespan of the schedule it constructs.

### Why priority queues are used

A priority queue retrieves the smallest keyed item efficiently. The task queue uses processing time as its key to implement SPT ordering; the crane queue uses availability time to examine cranes in earliest-available order. For each task, the algorithm checks all cranes because a crane that is available sooner may have to wait longer for spatial feasibility. It then rebuilds the crane queue from the updated availability times so that each crane appears exactly once.

The implementation searches integer start times from a crane's availability until it finds a feasible time. Under the stated assumptions, a feasible time exists: once all currently assigned tasks have ended, no task remains to create a spatial conflict. If there are $N$ tasks, $m$ cranes, and total processing time $P=\sum_{j\in J}p_j$, a conservative bound for this direct search is $O(N^2mP)$ time: each task checks each crane, each candidate time may inspect up to $N$ assignments, and the search horizon is bounded by the finite schedule length. The task and crane priority queues require $O(N+m)$ additional storage; the assignments and schedule require $O(N)$ storage.

### What to examine

The schedule and Gantt chart show when and where tasks are assigned. Crane utilization is each crane's busy time divided by the schedule makespan; system efficiency is total processing time divided by the total crane-time capacity. These are descriptive metrics, not guarantees of solution quality. A workload-based lower bound is also reported in the analysis: it can help contextualize the makespan, but the difference from that bound is not an optimality gap unless the bound is known to equal the optimum.


In [ ]:
import heapq
import time
from dataclasses import dataclass, field
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch, Rectangle

plt.style.use("default")

In [ ]:
@dataclass
class Bay:
    """A vessel bay represented as a processing task at a berth position."""

    vessel_id: int
    bay_id: int
    position: int
    processing_time: int
    priority: int = 1


@dataclass
class Vessel:
    """A vessel and its bay tasks."""

    vessel_id: int
    arrival_time: int
    due_date: int
    priority: int = 1
    bays: List[Bay] = field(default_factory=list)


@dataclass
class Crane:
    """A quay crane with its initial position and current availability state."""

    crane_id: int
    initial_position: int
    current_position: int = 0
    available_time: int = 0


@dataclass
class Assignment:
    """A task assigned to a crane over a half-open time interval."""

    crane_id: int
    vessel_id: int
    bay_id: int
    start_time: int
    end_time: int
    position: int
    processing_time: int


class EACSPTHeuristic:
    """Construct a crane schedule using SPT task order and earliest feasible starts."""

    def __init__(
        self,
        vessels: List[Vessel],
        cranes: List[Crane],
        clearance_distance: int = 2,
    ):
        if clearance_distance < 0:
            raise ValueError("Clearance distance must be nonnegative.")
        if len({crane.crane_id for crane in cranes}) != len(cranes):
            raise ValueError("Crane IDs must be unique.")
        if len({vessel.vessel_id for vessel in vessels}) != len(vessels):
            raise ValueError("Vessel IDs must be unique.")

        self.vessels = vessels
        self.cranes = cranes
        self.clearance_distance = clearance_distance
        self.crane_by_id = {crane.crane_id: crane for crane in cranes}
        self.crane_rank = {
            crane.crane_id: rank
            for rank, crane in enumerate(
                sorted(cranes, key=lambda item: (item.initial_position, item.crane_id))
            )
        }

        for crane in self.cranes:
            crane.current_position = crane.initial_position
            crane.available_time = 0

        self.all_bays = [bay for vessel in vessels for bay in vessel.bays]
        if any(bay.processing_time <= 0 for bay in self.all_bays):
            raise ValueError("Every bay must have a positive processing time.")

        self.assignments: List[Assignment] = []
        self.schedule: Dict[int, List[Assignment]] = {}
        self.makespan = 0

    def check_spatial_feasibility(
        self, crane: Crane, bay: Bay, start_time: int
    ) -> bool:
        """Check clearance and crane order over the entire candidate interval."""
        end_time = start_time + bay.processing_time

        for assignment in self.assignments:
            overlaps = (
                start_time < assignment.end_time
                and assignment.start_time < end_time
            )
            if not overlaps:
                continue

            if abs(bay.position - assignment.position) < self.clearance_distance:
                return False

            other_crane = self.crane_by_id[assignment.crane_id]
            if self.crane_rank[crane.crane_id] < self.crane_rank[other_crane.crane_id]:
                if bay.position > assignment.position:
                    return False
            elif bay.position < assignment.position:
                return False

        return True

    def check_temporal_feasibility(
        self, crane: Crane, bay: Bay, start_time: int
    ) -> bool:
        """Check that the crane is available when the task starts."""
        return start_time >= crane.available_time

    def find_earliest_feasible_time(self, crane: Crane, bay: Bay) -> int:
        """Search integer start times from crane availability for a feasible slot."""
        start_time = crane.available_time

        while True:
            if (
                self.check_temporal_feasibility(crane, bay, start_time)
                and self.check_spatial_feasibility(crane, bay, start_time)
            ):
                return start_time
            start_time += 1

    def solve(self, time_limit: int = 300) -> Dict:
        """Build and return a schedule using the EAC-SPT priority rules."""
        del time_limit  # Retained in the interface; this constructive method has no solver limit.
        if not self.all_bays:
            raise ValueError("At least one bay task is required.")
        if not self.cranes:
            raise ValueError("At least one crane is required.")

        print("EAC-SPT heuristic: constructing a schedule")
        start_time_total = time.time()

        task_queue = []
        for input_order, bay in enumerate(self.all_bays):
            heapq.heappush(
                task_queue, (bay.processing_time, input_order, bay)
            )
        print(f"Task priority queue: {len(task_queue)} tasks, shortest processing time first")

        crane_queue = []
        for crane in self.cranes:
            heapq.heappush(
                crane_queue, (crane.available_time, crane.crane_id, crane)
            )
        print(f"Crane availability queue: {len(crane_queue)} cranes")

        self.assignments = []
        assignment_count = 0

        while task_queue:
            _, _, bay = heapq.heappop(task_queue)
            crane_candidates = []
            best_crane = None
            best_start_time = float("inf")

            while crane_queue:
                _, _, crane = heapq.heappop(crane_queue)
                crane_candidates.append(crane)
                feasible_time = self.find_earliest_feasible_time(crane, bay)

                if feasible_time < best_start_time:
                    best_start_time = feasible_time
                    best_crane = crane

            if best_crane is None:
                raise RuntimeError("No feasible crane was found for a bay task.")

            assignment = Assignment(
                crane_id=best_crane.crane_id,
                vessel_id=bay.vessel_id,
                bay_id=bay.bay_id,
                start_time=best_start_time,
                end_time=best_start_time + bay.processing_time,
                position=bay.position,
                processing_time=bay.processing_time,
            )
            self.assignments.append(assignment)
            best_crane.available_time = assignment.end_time
            best_crane.current_position = bay.position
            assignment_count += 1

            print(
                f"Assignment {assignment_count}: Crane {best_crane.crane_id} "
                f"to Vessel {bay.vessel_id}, Bay {bay.bay_id}; "
                f"time {assignment.start_time}–{assignment.end_time}, "
                f"position {bay.position}"
            )

            # Rebuilding from the crane states keeps one current queue entry per crane.
            crane_queue = [
                (crane.available_time, crane.crane_id, crane)
                for crane in crane_candidates
            ]
            heapq.heapify(crane_queue)

        self.makespan = max(assignment.end_time for assignment in self.assignments)
        self.schedule = {crane.crane_id: [] for crane in self.cranes}
        for assignment in self.assignments:
            self.schedule[assignment.crane_id].append(assignment)
        for assignments in self.schedule.values():
            assignments.sort(key=lambda item: item.start_time)

        assert len(self.assignments) == len(self.all_bays)
        end_time_total = time.time()
        computation_time = end_time_total - start_time_total

        print("Schedule construction complete")
        print(f"Total assignments: {len(self.assignments)}")
        print(f"Makespan: {self.makespan} time units")
        print(f"Computation time: {computation_time:.6f} seconds")

        return {
            "assignments": self.assignments,
            "schedule": self.schedule,
            "makespan": self.makespan,
            "computation_time": computation_time,
        }

    def calculate_performance_metrics(self) -> Dict:
        """Calculate workload, crane utilization, and vessel completion metrics."""
        if not self.assignments:
            return {}

        total_processing_time = sum(bay.processing_time for bay in self.all_bays)
        crane_utilization = {}
        for crane in self.cranes:
            crane_assignments = self.schedule.get(crane.crane_id, [])
            busy_time = sum(item.processing_time for item in crane_assignments)
            crane_utilization[crane.crane_id] = {
                "busy_time": busy_time,
                "total_time": self.makespan,
                "utilization_rate": busy_time / self.makespan,
                "num_assignments": len(crane_assignments),
            }

        vessel_completion = {}
        for vessel in self.vessels:
            vessel_assignments = [
                item for item in self.assignments
                if item.vessel_id == vessel.vessel_id
            ]
            vessel_completion[vessel.vessel_id] = (
                max(item.end_time for item in vessel_assignments)
                if vessel_assignments
                else 0
            )

        total_capacity = len(self.cranes) * self.makespan
        return {
            "makespan": self.makespan,
            "total_assignments": len(self.assignments),
            "crane_utilization": crane_utilization,
            "vessel_completion": vessel_completion,
            "total_processing_time": total_processing_time,
            "system_efficiency": total_processing_time / total_capacity,
        }

    def visualize_solution(self, solution: Dict) -> None:
        """Plot the crane schedule, assignment order, utilization, and summary."""
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle("EAC-SPT Crane Assignment and Schedule", fontsize=16, fontweight="bold")

        colors = plt.cm.Set3(np.linspace(0, 1, max(len(self.vessels), 1)))
        color_by_vessel = {
            vessel.vessel_id: colors[index]
            for index, vessel in enumerate(self.vessels)
        }

        ax1.set_title("Crane Schedule (Gantt Chart)", fontweight="bold")
        ax1.set_xlabel("Time (time units)")
        ax1.set_ylabel("Crane ID")
        for crane_id, assignments in solution["schedule"].items():
            for assignment in assignments:
                ax1.add_patch(
                    Rectangle(
                        (assignment.start_time, crane_id - 0.4),
                        assignment.processing_time,
                        0.8,
                        facecolor=color_by_vessel[assignment.vessel_id],
                        edgecolor="black",
                    )
                )
                ax1.text(
                    assignment.start_time + assignment.processing_time / 2,
                    crane_id,
                    f"V{assignment.vessel_id}B{assignment.bay_id}",
                    ha="center",
                    va="center",
                    fontsize=8,
                )
        ax1.set_xlim(-0.5, max(self.makespan, 1) + 0.5)
        ax1.set_ylim(-0.5, max(len(self.cranes) - 0.5, 0.5))
        ax1.set_yticks([crane.crane_id for crane in self.cranes])
        ax1.grid(True, alpha=0.3)
        if self.vessels:
            ax1.legend(
                handles=[
                    Patch(facecolor=color_by_vessel[vessel.vessel_id], label=f"Vessel {vessel.vessel_id}")
                    for vessel in self.vessels
                ],
                title="Vessel",
                loc="upper right",
            )

        ax2.set_title("Task Assignment Order", fontweight="bold")
        ax2.set_xlabel("Time (time units)")
        ax2.set_ylabel("Assignment sequence")
        for order, assignment in enumerate(self.assignments):
            ax2.add_patch(
                Rectangle(
                    (assignment.start_time, order - 0.3),
                    assignment.processing_time,
                    0.6,
                    facecolor=color_by_vessel[assignment.vessel_id],
                    edgecolor="black",
                )
            )
            ax2.text(
                assignment.start_time + assignment.processing_time / 2,
                order,
                f"C{assignment.crane_id}–V{assignment.vessel_id}B{assignment.bay_id}",
                ha="center",
                va="center",
                fontsize=7,
            )
        ax2.set_xlim(-0.5, max(self.makespan, 1) + 0.5)
        ax2.set_ylim(-0.5, max(len(self.assignments) - 0.5, 0.5))
        ax2.grid(True, alpha=0.3)

        metrics = self.calculate_performance_metrics()
        ax3.set_title("Crane Utilization", fontweight="bold")
        ax3.set_xlabel("Crane ID")
        ax3.set_ylabel("Utilization rate")
        crane_ids = list(metrics["crane_utilization"])
        utilization_rates = [
            metrics["crane_utilization"][crane_id]["utilization_rate"]
            for crane_id in crane_ids
        ]
        bars = ax3.bar(crane_ids, utilization_rates, color="lightcoral", edgecolor="darkred")
        ax3.set_ylim(0, 1)
        ax3.set_xticks(crane_ids)
        for bar, rate in zip(bars, utilization_rates):
            ax3.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{rate:.1%}",
                ha="center",
                va="bottom",
            )
        ax3.grid(True, alpha=0.3, axis="y")

        ax4.set_title("Schedule Metrics", fontweight="bold")
        ax4.axis("off")
        summary_lines = [
            f"Makespan: {metrics['makespan']} time units",
            f"Assigned tasks: {metrics['total_assignments']}",
            f"Total processing time: {metrics['total_processing_time']} time units",
            f"System efficiency: {metrics['system_efficiency']:.1%}",
            "",
            "Crane utilization:",
        ]
        for crane_id, utilization in metrics["crane_utilization"].items():
            summary_lines.append(
                f"  Crane {crane_id}: {utilization['utilization_rate']:.1%} "
                f"({utilization['num_assignments']} tasks)"
            )
        summary_lines.extend(["", "Vessel completion times:"])
        for vessel_id, completion_time in metrics["vessel_completion"].items():
            summary_lines.append(f"  Vessel {vessel_id}: {completion_time} time units")
        ax4.text(
            0.05,
            0.95,
            "\n".join(summary_lines),
            transform=ax4.transAxes,
            va="top",
            family="monospace",
            bbox={"boxstyle": "round,pad=0.5", "facecolor": "lightblue", "alpha": 0.8},
        )

        plt.tight_layout()
        plt.show()

    def print_solution_summary(self, solution: Dict) -> None:
        """Print the instance, crane schedules, and descriptive metrics."""
        print("\n" + "=" * 72)
        print("EAC-SPT CRANE ASSIGNMENT AND SCHEDULE")
        print("=" * 72)
        print(f"Vessels: {len(self.vessels)}")
        print(f"Cranes: {len(self.cranes)}")
        print(f"Bay tasks: {len(self.all_bays)}")
        print(f"Clearance distance: {self.clearance_distance} position units")
        print(f"Computation time: {solution['computation_time']:.6f} seconds")
        print(f"Makespan: {solution['makespan']} time units")

        for crane_id, assignments in solution["schedule"].items():
            print(f"\nCrane {crane_id}:")
            if not assignments:
                print("  No assignments")
            for assignment in assignments:
                print(
                    f"  Time {assignment.start_time}–{assignment.end_time}: "
                    f"Vessel {assignment.vessel_id}, Bay {assignment.bay_id}, "
                    f"position {assignment.position}"
                )

        metrics = self.calculate_performance_metrics()
        average_utilization = np.mean(
            [item["utilization_rate"] for item in metrics["crane_utilization"].values()]
        )
        print("\nPerformance metrics:")
        print(f"System efficiency: {metrics['system_efficiency']:.1%}")
        print(f"Average crane utilization: {average_utilization:.1%}")
        print(f"Total processing time: {metrics['total_processing_time']} time units")
        print("=" * 72)

### Example instance

The example contains six vessels, each with two bay tasks, and four cranes. Each bay is represented by a processing time, berth position, and priority value. Processing time and position are used by the scheduling rule and spatial checks; the stored priority value is not part of EAC-SPT. The vessels' arrival times and due dates are also stored but, as described in the scope, do not affect this schedule.

In [ ]:
print("Creating the six-vessel, four-crane example")

vessels = []
bay_configs = [
    [(1, 15, 1), (2, 25, 3)],
    [(2, 35, 2), (1, 45, 4)],
    [(3, 55, 2), (2, 65, 1)],
    [(1, 75, 3), (2, 85, 2)],
    [(2, 95, 1), (3, 105, 3)],
    [(4, 115, 2), (1, 125, 2)],
]

for vessel_id, config in enumerate(bay_configs):
    bays = [
        Bay(
            vessel_id=vessel_id,
            bay_id=bay_id,
            position=position,
            processing_time=processing_time,
            priority=priority,
        )
        for bay_id, (processing_time, position, priority) in enumerate(config)
    ]
    vessels.append(
        Vessel(
            vessel_id=vessel_id,
            arrival_time=0,
            due_date=20,
            bays=bays,
        )
    )

cranes = [
    Crane(crane_id=0, initial_position=0),
    Crane(crane_id=1, initial_position=30),
    Crane(crane_id=2, initial_position=60),
    Crane(crane_id=3, initial_position=90),
]

all_processing_times = [
    bay.processing_time for vessel in vessels for bay in vessel.bays
]
print(f"Vessels: {len(vessels)}")
print(f"Bay tasks: {len(all_processing_times)}")
print(f"Cranes: {len(cranes)}")
print(f"Processing-time range: {min(all_processing_times)}–{max(all_processing_times)} time units")
print(f"Mean processing time: {np.mean(all_processing_times):.2f} time units")
print(f"Total processing time: {sum(all_processing_times)} time units")
print("Clearance distance: 2 position units")

### Construct and inspect the schedule

The solver processes the shortest remaining task first. For each task, it evaluates every crane's earliest spatially feasible start time and selects the earliest one. The printed assignment sequence therefore records the construction order; the crane schedules and Gantt chart show the resulting time intervals.

In [ ]:
heuristic = EACSPTHeuristic(vessels, cranes, clearance_distance=2)
solution = heuristic.solve()

heuristic.print_solution_summary(solution)
heuristic.visualize_solution(solution)

### Performance analysis

Crane utilization describes the fraction of the makespan spent processing tasks. System efficiency aggregates busy time across all cranes. Vessel completion time is the end of the last scheduled bay task for that vessel. These statistics describe the constructed schedule; none establishes that it is optimal.

A simple lower bound on any makespan is the greater of the average workload per crane and the duration of the longest individual task. The first term follows because $m$ cranes can process at most $mC_{\max}$ total time units by makespan $C_{\max}$; the second follows because even one task cannot finish faster than its own processing time. Spatial restrictions can make the actual optimum larger. Consequently, the difference between the heuristic makespan and this lower bound is a bound-based comparison, not necessarily the heuristic's distance from the true optimum.

In [ ]:
metrics = heuristic.calculate_performance_metrics()

print("EAC-SPT schedule analysis")
print(f"Computation time: {solution['computation_time']:.6f} seconds")
print(f"Makespan: {metrics['makespan']} time units")
print(f"System efficiency: {metrics['system_efficiency']:.1%}")

print("\nCrane utilization:")
utilization_rates = []
for crane_id, utilization in metrics["crane_utilization"].items():
    utilization_rates.append(utilization["utilization_rate"])
    print(
        f"Crane {crane_id}: {utilization['utilization_rate']:.1%} "
        f"({utilization['busy_time']}/{utilization['total_time']} time units)"
    )
print(f"Average utilization: {np.mean(utilization_rates):.1%}")
print(f"Utilization standard deviation: {np.std(utilization_rates):.3f}")

completion_times = list(metrics["vessel_completion"].values())
print("\nVessel completion:")
print(f"Mean completion time: {np.mean(completion_times):.2f} time units")
print(
    "Completion-time range: "
    f"{min(completion_times)}–{max(completion_times)} time units"
)
print(f"Completion-time variance: {np.var(completion_times):.2f}")

total_processing = sum(bay.processing_time for bay in heuristic.all_bays)
longest_task = max(bay.processing_time for bay in heuristic.all_bays)
workload_lower_bound = max(total_processing / len(cranes), longest_task)
difference_from_bound = metrics["makespan"] - workload_lower_bound

print("\nWorkload lower-bound comparison:")
print(f"Workload-based lower bound: {workload_lower_bound:.2f} time units")
print(f"Heuristic makespan: {metrics['makespan']} time units")
print(f"Difference from lower bound: {difference_from_bound:.2f} time units")

### Strengths, limitations, and appropriate use

EAC-SPT is a transparent constructive rule: a priority queue makes shortest-processing-time task selection explicit, and the feasibility checks account for crane availability, berth clearance, and non-crossing order. Its greedy decisions can produce a feasible schedule without enumerating every possible assignment and start-time combination.

The method has no general optimality guarantee. A locally earliest feasible assignment can constrain later tasks, and solution quality depends on the instance and the priority rule. The implementation's direct integer-time search and repeated checks against existing assignments can also become costly as the schedule grows; the complexity bound given above reflects this implementation rather than a universal performance guarantee. No comparison with an exact optimizer or empirical solution-quality guarantee is produced in this notebook.

This heuristic is useful when a transparent schedule-construction rule is the focus and a feasible schedule is more important than a proof of optimality. Applications that require arrival-time handling, travel times, additional operational constraints, or an optimality certificate need those requirements represented explicitly in the model. This notebook does not implement those extensions.